In [8]:
import os
os.chdir("/home/luhongxuan/FakeNews_IDS_Project/data_pipeline")
print(os.getcwd())
from collectors.fact_check import search_claims, parse_claims
import pandas as pd


# ── 試打一個查詢 ──────────────────────────────────────
raw = search_claims("COVID vaccine DNA")
print("原始回傳：")
#print(raw)
for clain in raw.get("claims", []):
    print(clain)

/home/luhongxuan/FakeNews_IDS_Project/data_pipeline
原始回傳：
{'text': '“SV40, a cancer causing sequence” was “put in the Covid Vaccine.”', 'claimant': 'Social media posts', 'claimDate': '2023-09-20T00:00:00Z', 'claimReview': [{'publisher': {'name': 'FactCheck.org', 'site': 'factcheck.org'}, 'url': 'https://www.factcheck.org/2023/10/scicheck-covid-19-vaccines-have-not-been-shown-to-alter-dna-cause-cancer/', 'title': 'COVID-19 Vaccines Have Not Been Shown to Alter DNA, Cause ...', 'textualRating': 'Misleading', 'languageCode': 'en'}]}
{'text': "A Swedish study shows that Pfizer's COVID-19 vaccine changes recipients' DNA. AP'S ASSESSMENT", 'claimant': 'social media users', 'claimDate': '2022-07-20T17:25:00Z', 'claimReview': [{'publisher': {'name': 'AP News', 'site': 'apnews.com'}, 'url': 'https://apnews.com/article/Fact-Check-COVID-Vaccine-Sweden-Study-986569377766', 'title': 'Swedish study on COVID vaccines and DNA misinterpreted', 'textualRating': 'Social media users are citing a study fro

In [12]:

for claim in raw.get("claims", []):
    print(claim)
    print(claim.keys())
    print(claim.get("claimReview", []))

{'text': '“SV40, a cancer causing sequence” was “put in the Covid Vaccine.”', 'claimant': 'Social media posts', 'claimDate': '2023-09-20T00:00:00Z', 'claimReview': [{'publisher': {'name': 'FactCheck.org', 'site': 'factcheck.org'}, 'url': 'https://www.factcheck.org/2023/10/scicheck-covid-19-vaccines-have-not-been-shown-to-alter-dna-cause-cancer/', 'title': 'COVID-19 Vaccines Have Not Been Shown to Alter DNA, Cause ...', 'textualRating': 'Misleading', 'languageCode': 'en'}]}
dict_keys(['text', 'claimant', 'claimDate', 'claimReview'])
[{'publisher': {'name': 'FactCheck.org', 'site': 'factcheck.org'}, 'url': 'https://www.factcheck.org/2023/10/scicheck-covid-19-vaccines-have-not-been-shown-to-alter-dna-cause-cancer/', 'title': 'COVID-19 Vaccines Have Not Been Shown to Alter DNA, Cause ...', 'textualRating': 'Misleading', 'languageCode': 'en'}]
{'text': "A Swedish study shows that Pfizer's COVID-19 vaccine changes recipients' DNA. AP'S ASSESSMENT", 'claimant': 'social media users', 'claimDat

In [14]:
from datetime import datetime
date = "20210201T120000Z"
dt = datetime.strptime(date, "%Y%m%dT%H%M%SZ")
print(dt)

2021-02-01 12:00:00


In [10]:
import httpx
response = httpx.get("https://api.gdeltproject.org/api/v2/doc/doc", params={"query": "COVID", "mode": "artlist", "maxrecords": 5, "format": "json", "sort": "DateDesc"})
print(response.status_code)
print(response.text)

ConnectTimeout: _ssl.c:983: The handshake operation timed out

In [24]:
import httpx

BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

async def fetch_news(query: str, max_results: int = 20) -> list[dict]:
    params = {
        "query"     : query,
        "mode"      : "artlist",
        "maxrecords": max_results,
        "format"    : "json",
        "sort"      : "DateDesc",
    }

    async with httpx.AsyncClient(timeout=30) as client:
        response = await client.get(BASE_URL, params=params)
        response.raise_for_status()
        data = response.json()

    articles = []
    seen_urls = set()   # 去重用

    for article in data.get("articles", []):
        url = article.get("url")

        # 去重
        if url in seen_urls:
            continue
        seen_urls.add(url)

        articles.append({
            "title"  : article.get("title"),
            "url"    : url,
            "source" : article.get("domain"),
            "date"   : article.get("seendate"),
            "country": article.get("sourcecountry"),
        })

    # 時間排序（新到舊）
    articles.sort(key=lambda x: x["date"] or "", reverse=True)

    return articles[:max_results]

In [25]:
data = await fetch_news("COVID")


/tmp/ipykernel_39573/417120499.py:1: RuntimeWarning: coroutine 'fetch_news' was never awaited
  data = await fetch_news("COVID")


In [31]:
print(data)
print(data[0].keys())
for e in data[0].keys():
    print(e, data[0][e])

[{'title': 'Inside the world of ultra - luxury wedding cakes', 'url': 'https://www.advocateanddemocrat.com/news/national/article_d0ab6618-d1e3-557c-bd82-cc87776d5da9.html', 'source': 'advocateanddemocrat.com', 'date': '20260421T061500Z', 'country': 'United States'}, {'title': 'I built a proper old - school pub in my garden - now Im selling up', 'url': 'https://www.dailymail.com/money/mortgageshome/article-15013931/I-built-proper-old-school-pub-garden-Im-selling-up.html', 'source': 'dailymail.com', 'date': '20260421T061500Z', 'country': 'United States'}, {'title': 'Woolies hits back at  misguided  watchdog price claims', 'url': 'https://www.southcoastregister.com.au/story/9229106/woolies-hits-back-at-misguided-watchdog-price-claims/', 'source': 'southcoastregister.com.au', 'date': '20260421T061500Z', 'country': 'Australia'}, {'title': 'Ironfest triumphant return announces 2027  Gilded Age  theme | Lithgow Mercury', 'url': 'https://www.lithgowmercury.com.au/story/9228450/ironfests-triump

In [ ]:
claims = parse_claims(raw)
df = pd.DataFrame(claims)

print(f"找到 {len(df)} 筆查核結果")
df